In [ ]:

import random

In [ ]:
class Board:
    LINES = [(0,1,2), (3,4,5), (6,7,8),   # rows
             (0,3,6), (1,4,7), (2,5,8),   # columns
             (0,4,8), (2,4,6)]            # diagonals

    def __init__(self):
        self.reset()

    def reset(self):
        self.cells = ["."] * 9

    def available_moves(self):
        return [i for i, c in enumerate(self.cells) if c == "."]

    def make_move(self, pos, mark):
        if self.cells[pos] != ".":
            raise ValueError(f"Square {pos} is taken")
        self.cells[pos] = mark

    def winner(self):
        for a, b, c in self.LINES:
            if self.cells[a] != "." and self.cells[a] == self.cells[b] == self.cells[c]:
                return self.cells[a]
        return None

    def is_full(self):
        return "." not in self.cells

    def is_over(self):
        return self.winner() is not None or self.is_full()

    def to_key(self):
        return "".join(self.cells)

    def display(self):
        for r in range(3):
            print(" ".join(self.cells[3*r : 3*r+3]))
    print()

In [ ]:
class RandomPlayer:
    def __init__(self, mark="O"):
        self.mark = mark

    def choose_move(self, board):
        return random.choice(board.available_moves())

In [ ]:
class Environment:
    WIN_REWARD = 10
    LOSS_REWARD = -10
    DRAW_REWARD = 0
    MOVE_REWARD = 0

    def __init__(self, board, opponent):
        self.board = board
        self.opponent = opponent
        self.agent_mark = "X"
        self.result = None

    def reset(self, agent_starts=True):
        self.board.reset()
        self.result = None
        if not agent_starts:
            move = self.opponent.choose_move(self.board)
            self.board.make_move(move, self.opponent.mark)
        return self.board.to_key()

    def step(self, action):
        # 1. Agent's move
        self.board.make_move(action, self.agent_mark)
        if self.board.winner() == self.agent_mark:
            self.result = "win"
            return self.board.to_key(), self.WIN_REWARD, True
        if self.board.is_full():
            self.result = "draw"
            return self.board.to_key(), self.DRAW_REWARD, True

        # 2. Opponent's reply
        move = self.opponent.choose_move(self.board)
        self.board.make_move(move, self.opponent.mark)
        if self.board.winner() == self.opponent.mark:
            self.result = "loss"
            return self.board.to_key(), self.LOSS_REWARD, True
        if self.board.is_full():
            self.result = "draw"
            return self.board.to_key(), self.DRAW_REWARD, True

        # 3. Game continues
        return self.board.to_key(), self.MOVE_REWARD, False

    def get_result(self):
        return self.result

In [ ]:
class Agent:
    def __init__(self, epsilon=1.0, alpha=0.1, gamma=0.9, epsilon_min=0.05, epsilon_decay=0.999):
      self.q_table = {}
      self.epsilon = epsilon
      self.alpha = alpha
      self.gamma = gamma
      self.epsilon_min = epsilon_min        # never explore less than 5%
      self.epsilon_decay = epsilon_decay    # shrink a little after each game

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def get_q(self, state):
        if state not in self.q_table:
            self.q_table[state] = [0.0] * 9
        return self.q_table[state]

    def legal_actions(self, state):
        return [i for i, c in enumerate(state) if c == "."]

    def best_action(self, state):
        """The empty square with the highest Q. If several tie, pick one at random."""
        q = self.get_q(state)
        actions = self.legal_actions(state)
        best_q = max(q[a] for a in actions)
        best = [a for a in actions if q[a] == best_q]
        return random.choice(best)

    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.choice(self.legal_actions(state))   # explore
        return self.best_action(state)                        # exploit

    def update(self, state, action, reward, next_state, done):
      q = self.get_q(state)

      if done:
          # Game over: no next move, so the target is just the reward
          target = reward
      else:
          # Reward now + discounted value of the best next move
          next_q = self.get_q(next_state)
          best_next = max(next_q[a] for a in self.legal_actions(next_state))
          target = reward + self.gamma * best_next

      # Move the guess a small step toward the target
      q[action] += self.alpha * (target - q[action])

In [ ]:
class Game:
    def __init__(self, agent, env, num_episodes=10000, report_every=1000):
        self.agent = agent
        self.env = env
        self.num_episodes = num_episodes
        self.report_every = report_every
        self.wins = self.losses = self.draws = 0
        self.total_wins = self.total_losses = self.total_draws = 0
        self.seen_states = set()                      # NEW

    def play_episode(self):
        state = self.env.reset(agent_starts=random.choice([True, False]))
        done = False
        while not done:
            self.seen_states.add(state)               # NEW
            action = self.agent.choose_action(state)
            next_state, reward, done = self.env.step(action)
            self.agent.update(state, action, reward, next_state, done)
            state = next_state
        self.agent.decay_epsilon()
        return self.env.get_result()

    def record(self, result):
        if result == "win":
            self.wins += 1; self.total_wins += 1
        elif result == "loss":
            self.losses += 1; self.total_losses += 1
        else:
            self.draws += 1; self.total_draws += 1

    def report(self, episode):
        start = episode - self.report_every + 1
        win_pct = 100 * self.wins / self.report_every
        print(f"Episodes {start:5}-{episode:5} | W {self.wins} L {self.losses} "
              f"D {self.draws} | win {win_pct:.1f}% | eps {self.agent.epsilon:.3f}")
        self.wins = self.losses = self.draws = 0

    def train(self):
        for episode in range(1, self.num_episodes + 1):
            result = self.play_episode()
            self.record(result)
            if episode % self.report_every == 0:
                self.report(episode)
        print(f"Total: W {self.total_wins} L {self.total_losses} D {self.total_draws}")
        print(f"Different states seen: {len(self.seen_states)}")   # NEW

    def evaluate(self, num_games=10000):
        """Play without exploring or learning, and count results by who started."""
        saved = self.agent.epsilon
        self.agent.epsilon = 0.0                  # always play the best move
        results = {True:  {"win": 0, "loss": 0, "draw": 0},
                    False: {"win": 0, "loss": 0, "draw": 0}}

        for _ in range(num_games):
            agent_starts = random.choice([True, False])
            state = self.env.reset(agent_starts=agent_starts)
            done = False
            while not done:
                action = self.agent.choose_action(state)
                state, reward, done = self.env.step(action)   # no update() = no learning
            results[agent_starts][self.env.get_result()] += 1

        self.agent.epsilon = saved                # put epsilon back
        for starts, name in ((True, "Agent starts"), (False, "Opponent starts")):
            r = results[starts]
            n = sum(r.values())
            print(f"{name:16}| W {r['win']:5} L {r['loss']:5} D {r['draw']:5} | "
                  f"win {100*r['win']/n:.1f}% loss {100*r['loss']/n:.1f}%")

In [ ]:
agent = Agent()
env = Environment(Board(), RandomPlayer("O"))
game = Game(agent, env, num_episodes=20000000
, report_every=10000)
game.train()

Episodes     1-10000 | W 7428 L 1521 D 1051 | win 74.3% | eps 0.050
Episodes 10001-20000 | W 8328 L 550 D 1122 | win 83.3% | eps 0.050
Episodes 20001-30000 | W 8356 L 516 D 1128 | win 83.6% | eps 0.050
Episodes 30001-40000 | W 8490 L 467 D 1043 | win 84.9% | eps 0.050
Episodes 40001-50000 | W 8527 L 459 D 1014 | win 85.3% | eps 0.050
Episodes 50001-60000 | W 8588 L 451 D 961 | win 85.9% | eps 0.050
Episodes 60001-70000 | W 8630 L 419 D 951 | win 86.3% | eps 0.050
Episodes 70001-80000 | W 8684 L 406 D 910 | win 86.8% | eps 0.050
Episodes 80001-90000 | W 8599 L 446 D 955 | win 86.0% | eps 0.050
Episodes 90001-100000 | W 8684 L 371 D 945 | win 86.8% | eps 0.050
Episodes 100001-110000 | W 8662 L 370 D 968 | win 86.6% | eps 0.050
Episodes 110001-120000 | W 8792 L 355 D 853 | win 87.9% | eps 0.050
Episodes 120001-130000 | W 8800 L 341 D 859 | win 88.0% | eps 0.050
Episodes 130001-140000 | W 8752 L 347 D 901 | win 87.5% | eps 0.050
Episodes 140001-150000 | W 8714 L 381 D 905 | win 87.1% | eps

In [ ]:
import json
from google.colab import files

with open("qtable.json", "w") as f:
    json.dump(agent.q_table, f)

print("Saved", len(agent.q_table), "boards")
files.download("qtable.json")

Saved 4520 boards


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from IPython.display import HTML, display

def board_html(state, q):
    """One board as a 3x3 table: marks on taken squares, Q-values on empty ones."""
    actions = [i for i, c in enumerate(state) if c == "."]
    best_q = max(q[a] for a in actions)
    html = '<table style="border-collapse:collapse;margin:auto">'
    for r in range(3):
        html += "<tr>"
        for c in range(3):
            i = 3 * r + c
            bg = "#ffffff"
            if state[i] == "X":
                text = '<b style="color:#d62828;font-size:22px">X</b>'
            elif state[i] == "O":
                text = '<b style="color:#1f5fd6;font-size:22px">O</b>'
            else:
                text = f"{q[i]:.2f}"
                if q[i] == best_q:
                    bg = "#b7f0b1"          # best action: green
                    text = f"<b>{text}</b>"
            html += (f'<td style="width:56px;height:56px;border:1px solid #999;'
                     f'text-align:center;font-family:monospace;color:#222;'
                     f'background:{bg}">{text}</td>')
        html += "</tr>"
    return html + "</table>"


def show_q_table(q_table, moves=None, limit=12):
    """Show up to `limit` boards. moves = how many marks are already on the board."""
    shown = 0
    html = ""
    for state, q in q_table.items():
        actions = [i for i, c in enumerate(state) if c == "."]
        if not actions:
            continue                                    # full board: nothing to choose
        if moves is not None and 9 - len(actions) != moves:
            continue                                    # not the number of moves we want
        if all(q[a] == 0 for a in actions):
            continue                                    # never learned anything here
        html += (f'<div style="display:inline-block;margin:8px;padding:6px;'
                 f'background:#fff;color:#222;font-family:monospace;text-align:center">'
                 f'{state}<br>{board_html(state, q)}</div>')
        shown += 1
        if shown == limit:
            break
    display(HTML(html))
    print(f"Showing {shown} boards (Q-table has {len(q_table)} in total)")

In [ ]:
show_q_table(agent.q_table, moves=0)            # the empty board: the agent's favorite first move
show_q_table(agent.q_table, moves=1)            # boards where O played first
show_q_table(agent.q_table, moves=3, limit=20)  # mid-game boards, 20 of them

7.40,6.47,6.22
6.96,7.31,5.30
7.95,6.79,7.50


Showing 1 boards (Q-table has 4303 in total)


6.23,O,6.92
5.33,7.56,6.31
5.94,4.67,6.89
O,3.43,6.84
3.55,7.43,6.10
4.63,5.81,5.57
5.59,5.93,6.26
5.19,7.36,3.74
5.54,3.87,O
5.60,4.39,5.28
3.82,O,3.42


Showing 9 boards (Q-table has 4303 in total)


4.80,O,O
0.31,0.54,X
-1.90,-0.01,-1.00
O,0.00,7.94
0.00,0.74,0.00
0.00,O,X
-0.32,-1.00,-0.01
X,-1.65,-0.69
7.64,O,O
X,1.73,8.63
-0.63,O,1.80


Showing 20 boards (Q-table has 4303 in total)
